# 17 · 走通本机 `rag_project/`

> **学习目标**：把本机已经搭好的规范化 RAG 工程 [rag_project/](../../../../rag_project/) 跑通：看项目结构、调 CLI 的 stats / query / build、改 5 处 config 看效果变化。
>
> **预备**：notebook 16 跑过。
>
> **为什么重要**：notebook 16 是「演示骨架」，rag_project 是「工程化骨架」。理解一个**规范化项目**长什么样、模块如何拆分，是从 demo 写代码升级到能交付项目的关键一步。

In [ ]:
import subprocess, os, sys
from pathlib import Path

RAG_PROJECT = Path('f:/source/code/direction/rag/rag_project').resolve()
PYTHON = Path('D:/ProgramData/anaconda3/envs/rag/python.exe')
assert RAG_PROJECT.exists(), '请确认 rag_project 路径'
assert PYTHON.exists(),       '请确认 conda env rag 已建好'
print('RAG_PROJECT =', RAG_PROJECT)
print('PYTHON      =', PYTHON)

def sh(cmd: str, cwd: Path | str = RAG_PROJECT, timeout: int = 60) -> tuple[int, str]:
    """在 rag_project 目录下跑命令。强制 UTF-8 防 Windows GBK 翻车。"""
    print(f'$ (cwd={Path(cwd).name}) {cmd}')
    r = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True,
                        text=True, encoding='utf-8', errors='replace', timeout=timeout)
    out = ((r.stdout or '') + (r.stderr or '')).rstrip()
    if out:
        # 限制输出，避免淹掉 cell
        for line in out.splitlines()[:30]:
            print(line)
        if len(out.splitlines()) > 30:
            print(f'  ...（截断，剩 {len(out.splitlines()) - 30} 行）')
    return r.returncode, out

## 1. 看项目结构 —— 一个规范 RAG 工程应该长什么样

本机 rag_project 的目录约定（详见 [memory/rag_project_layout.md](file:///C:/Users/ril/.claude/projects/f--source-code-direction-rag/memory/rag_project_layout.md)）：

In [ ]:
# 顶层结构
for p in sorted(RAG_PROJECT.iterdir()):
    icon = '📁' if p.is_dir() else '📄'
    print(f'  {icon} {p.name}')

# src/rag 内部 —— 一个清爽的模块拆分
print('\nsrc/rag/:')
src_rag = RAG_PROJECT / 'src' / 'rag'
for p in sorted(src_rag.iterdir()):
    if p.is_dir():
        print(f'  📁 {p.name}/')
        for q in sorted(p.iterdir()):
            if q.suffix == '.py' and q.name != '__init__.py':
                print(f'      📄 {q.name}')
    elif p.suffix == '.py' and p.name != '__init__.py':
        print(f'  📄 {p.name}')

## 2. 看 config —— 所有「可调旋钮」都集中在一个 YAML 里

**为什么这么设计**：让「换 embedding 模型 / 改 chunk_size / 换向量库」**只动 config，不动代码**。这是工程化的最小一步。

In [ ]:
config_path = RAG_PROJECT / 'config' / 'default.yaml'
lines = config_path.read_text(encoding='utf-8').splitlines()
print(f'config: {config_path}  ({len(lines)} 行)\n')
# 标出关键 section
key_sections = ['loader:', 'chunking:', 'embedding:', 'vectorstore:', 'ingest:']
for i, line in enumerate(lines, 1):
    marker = '👉' if any(line.startswith(k) for k in key_sections) else '  '
    print(f'{marker} {i:3d} | {line}')

## 3. CLI —— 一个项目应有的 3 个最小入口

rag_project 暴露了 3 个子命令：
- `python -m rag.cli build` —— 完整重建索引
- `python -m rag.cli stats` —— 看当前索引状态
- `python -m rag.cli query "..."` —— 查询测试

**先看 help（不需要 Ollama，纯解析参数）**：

In [ ]:
code, _ = sh(f'"{PYTHON}" -m rag.cli --help', cwd=RAG_PROJECT / 'src')
assert code == 0, 'CLI 启动失败 —— 检查 src/rag/cli.py 是否能 import'

In [ ]:
# stats —— 看当前 collection 里有多少 chunk（如果还没 build 过，会报「collection 不存在」）
code, out = sh(f'"{PYTHON}" -m rag.cli stats', cwd=RAG_PROJECT / 'src')
if code != 0:
    print('\n(stats 失败属常见情况：可能还没 build；或者 chromadb 路径还没建立。继续走下面 query 即可)')

### 注意：build / query 需要 Ollama 在跑

`build` 调用 embedding（默认 Ollama `nomic-embed-text`）；`query` 也调 embedding。

**如果 ollama 没起，下面 2 个 cell 会失败 —— 是预期的**。这一段的学习价值在「看命令格式 + 看 config 与 CLI 的对应关系」，不在于实际跑出结果。

要看真实结果：在另一个终端执行 `ollama serve`，然后重跑下面这两格。

In [ ]:
import requests
try:
    requests.get('http://127.0.0.1:11434/api/tags', timeout=1).raise_for_status()
    OLLAMA_UP = True
    print('✅ Ollama 在线，可以跑 build / query')
except Exception:
    OLLAMA_UP = False
    print('⚠ Ollama 未启动 —— 跳过 build / query 实跑。建议：另起一个终端 `ollama serve` 后回来重跑。')

In [ ]:
if OLLAMA_UP:
    # query 一个示例（不需要 build 过，但需要 collection 存在）
    code, _ = sh(f'"{PYTHON}" -m rag.cli query "组件 1187 的参数" --top-k 3', cwd=RAG_PROJECT / 'src', timeout=120)
    # 失败也别 assert —— 若 collection 不存在或 Ollama 模型未拉，会失败
else:
    print('(skipped) Ollama 未在线')

## 4. 改 config 看效果 —— RAG 工程化的核心练习

**5 个最常调的 config 旋钮**（不需要真跑就能学到「在哪儿改」）：

In [ ]:
tweaks = [
    ('chunking.chunk_size',     '500 → 300 / 800', '小 chunk 召回粒度细但上下文片段化；大 chunk 反之'),
    ('chunking.chunk_overlap',  '80 → 0 / 200',     'overlap 解决「答案被切到 chunk 边界」的问题，但增加索引体积'),
    ('embedding.backend',       'ollama → local / openai', '换 embedding 后端，需重 build；本机 local 当前不可用（transformers 版本问题）'),
    ('embedding.ollama.model',  'nomic-embed-text → bge-m3', 'bge-m3 中文更好，但要 `ollama pull bge-m3`（网络）'),
    ('vectorstore.chroma.distance_fn', 'cosine → l2', '归一化向量下两者等价；未归一化时有差。详见 notebook 20'),
]
print(f'{"config 旋钮":<35} {"建议改成":<30} 影响')
print('-' * 100)
for path, change, effect in tweaks:
    print(f'{path:<35} {change:<30} {effect}')

In [ ]:
# 安全演练：复制 default.yaml -> experimental.yaml，改两个值，**不污染原 config**
import shutil, yaml

exp_cfg = RAG_PROJECT / 'config' / 'experimental.yaml'
if not exp_cfg.exists():
    shutil.copy(config_path, exp_cfg)

cfg = yaml.safe_load(exp_cfg.read_text(encoding='utf-8'))
old_size = cfg['chunking']['chunk_size']
cfg['chunking']['chunk_size'] = 300
cfg['chunking']['chunk_overlap'] = 50
exp_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

print(f'已建实验 config: {exp_cfg.name}')
print(f'  chunk_size:    {old_size} → 300')
print(f'  chunk_overlap: → 50')
print()
print('用这份 config build / query 的命令：')
print(f'  python -m rag.cli build --config config/experimental.yaml')
print(f'  python -m rag.cli query "..." --config config/experimental.yaml')
print('\n→ 自己跑两份不同 config 各 query 10 次，对比检索结果。这就是「最小 A/B 实验」。')

## 5. 读关键源码（5 个高优先级文件）

想真正看懂工程化 RAG 怎么写，按顺序读这 5 个：

In [ ]:
reading_list = [
    ('src/rag/cli.py',                       'CLI 入口，看 build / query / stats 各自怎么组装组件'),
    ('src/rag/config.py',                    'YAML + env loader + pydantic 校验，规范项目的标配'),
    ('src/rag/embeddings/factory.py',        '工厂模式：按 backend 字段返回正确的 Embedder 实例 —— 解耦点'),
    ('src/rag/pipeline/ingest.py',           '端到端 ingestion：loader → chunk → embed → store 的串联'),
    ('src/rag/vectorstore/chroma_store.py',  'Chroma 适配层：把 chromadb 的接口包成你的统一接口'),
]
for path, why in reading_list:
    p = RAG_PROJECT / path
    lines = p.read_text(encoding='utf-8').count('\n') + 1
    print(f'  [{lines:>4} 行] {path}')
    print(f'         → {why}')

## 深入思考

1. **为什么把 embedding 后端做成 factory？**
   - 解耦。`pipeline.ingest()` 不关心你用 Ollama 还是 OpenAI，只跟 `Embedder` 接口对话。换后端只动 config，不动业务代码。
2. **为什么 `ChromaStore` 还要再包一层而不直接用 `chromadb.Client`？**
   - 想未来换 Qdrant / Milvus 时，只换 Store 实现，pipeline 与 retrieval 代码不动。「抽象边界」就在「外部依赖 vs 业务逻辑」处。
3. **为什么有 `experimental.yaml` 这种东西？**
   - 每个改动都改 default.yaml 会污染主分支。**实验改在新 config 里**，跑通了再决定要不要进 default —— 这是 RAG A/B 实验的最低成本流程。
4. **rag_project 现在没用 langchain，是不是「不够先进」？**
   - 反过来。**最小依赖** = 出问题时栈追得到底；langchain 升级或重写 API 时你不会被拖。生产 RAG 大量从「全 langchain」回归到「关键组件自己写、langchain 只取若干组件」。
5. **如果让你升级这个 rag_project，下一步加什么？**
   - 投票顺序：(1) eval set + 自动跑分（notebook 24）→ (2) hybrid 检索（notebook 21）→ (3) reranker（notebook 22）→ (4) FastAPI/可观测（06 章）→ (5) 多模态（高级阶段）。

**改一改**：手动改 `config/experimental.yaml` 里的 `embedding.ollama.model`（若你有别的 Ollama 模型），先 `python -m rag.cli build --config config/experimental.yaml`，再 query 同一个问题，看结果差别。

## 自检 ✅

- [ ] 不看代码画出 rag_project 的目录树（顶层 + src/rag/ 下两级）。
- [ ] 解释「factory 模式在 rag_project 里解决了什么」。
- [ ] 默背 5 个最常调的 config 旋钮及影响。
- [ ] 给你一个 rag 项目，能在 10 分钟内画出「读 config → 加载 components → 跑端到端」的链路。
- [ ] 解释「为什么 ragflow / langchain / llama-index 这种大框架不一定适合所有项目」。

## 清理（可选）

```python
(RAG_PROJECT / 'config' / 'experimental.yaml').unlink(missing_ok=True)
```

## 下一步

→ [`18_doc_loaders_and_failure_modes.ipynb`](18_doc_loaders_and_failure_modes.ipynb)